# 05 - Implicit Alternating Least Squares (ALS) Collaborative Filtering
## E-Commerce Recommendation System

**Objective:** Train and evaluate matrix factorization using Implicit ALS on sparse user-item interaction matrix.

In [ ]:
import sys, pickle
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.collaborative_filtering import ALSRecommender
from src.features.build_features import build_sparse_interaction_matrix

train_user_item = pd.read_parquet(ROOT / 'data/processed/train.parquet')
print(f'Train events: {len(train_user_item):,}')


In [ ]:
# Build interaction matrix and train ALS
from src.features.build_features import aggregate_user_item_interactions
agg_train = aggregate_user_item_interactions(train_user_item)
mat, mappings = build_sparse_interaction_matrix(agg_train)

als = ALSRecommender(factors=64, regularization=0.1, iterations=20, random_state=42)
als.fit(mat, mappings)


In [ ]:
# Test recommendation for warm user
sample_visitor = list(mappings['visitorid_to_idx'].keys())[10]
recs = als.recommend(sample_visitor, top_n=10)
print(f'Personalized ALS recommendations for Visitor {sample_visitor}:')
for r, (iid, score) in enumerate(recs, 1):
    print(f'  Rank {r:2d}: Item {iid} (Predicted score: {score:.4f})')
